In [6]:
old_results_path = "../results/ICST_2026/varbugs_classified.json"
new_results_folder = "../sugarlyzer/results/"

import json
import os

with open(old_results_path, "r") as f:
    old_results = json.load(f)

new_results = []
for root, dirs, files in os.walk(new_results_folder):
    for file in files:
        if "varbugs" in os.path.join(root, file):
            with open(os.path.join(root, file), "r") as f:
                new_results.extend(json.load(f))
                # Process new_results and compare with old_results

from pathlib import Path

match = 0
not_match = 0
product_match = 0
transformation_match = 0

def get_line_number_from_new(result):
   if result.get('lineInputFile') is not None:
      return result['lineInputFile']
   else:
      return [result['originalAlarm']['line'], result['originalAlarm']['line']]
      
def get_description_from_new(result):
   if result.get('sanitizedDescription') is not None:
      return result['sanitizedDescription']
   else:
      return result['originalAlarm']['description']
   
for new in new_results:
  new['classification'] = 'UNKNOWN'
  new['old_id'] = None
  def pred(old, new):
    if old['origin'] in ['product', 'transformation']:
      if Path(old['input_file']).name == Path(new['originalAlarm']['fileLocation']).name:
         try:
            old_line = list(map(int, old['original_line'].split(':')))
            new_line = get_line_number_from_new(new)
            if old_line == new_line:
               print(f"Old line: {old_line}, New line: {new_line}")
               if old['sanitized_message'] == get_description_from_new(new):
                  print(f"Old description: {old['sanitized_message']}, New description: {get_description_from_new(new)}")
                  new['old_id'] = old['id']
                  new['classification'] = old['classification']
                  return True
         except ValueError as ve:
            print(f"ValueError: {ve} in new result, skipping line number comparison.")
            return False
    elif old['origin'] == 'family':
       return False
    else:
      raise ValueError("Unknown format in old results")

  
  if any(pred(old, new) for old in old_results):
    print(f"Match found!")
    if new['strategy'] == "product":
      product_match += 1
    elif new['strategy'] == "transformation":
      transformation_match += 1
    match += 1
  else:
    print("No match found.")
    not_match += 1

print(f"Total matches: {match}")
print(f"Total non-matches: {not_match}")
print(f"Product matches: {product_match}")
print(f"Transformation matches: {transformation_match}")

product = 1
transformation = 1
for new in new_results:
   if new['strategy'] == "product":
      new['id'] = "PRODUCT_" + str(product)
      product += 1
   elif new['strategy'] == "transformation":
      new['id'] = "TRANSFORMATION_" + str(transformation)
      transformation += 1


with open("new_classifications.json", "w") as f:
    json.dump(new_results, f, indent=4)

Old line: [34, 34], New line: [34, 34]
Old description: Potential leak of memory pointed to by 'buf', New description: Potential leak of memory pointed to by 'buf'
Match found!
No match found.
Old line: [46, 46], New line: [46, 46]
Old description: Potential leak of memory pointed to by 'clist', New description: Potential leak of memory pointed to by 'clist'
Match found!
No match found.
No match found.
Old line: [24, 24], New line: [24, 24]
Old description: Value stored to 'codec_hw_write' during its initialization is never read, New description: Value stored to 'codec_hw_write' during its initialization is never read
Match found!
Old line: [16, 16], New line: [16, 16]
Old description: Value stored to 'val' is never read, New description: Value stored to 'val' is never read
Match found!
Old line: [6, 6], New line: [6, 6]
Old description: Null pointer passed to 1st parameter expecting 'nonnull', New description: Null pointer passed to 1st parameter expecting 'nonnull'
Match found!
Old l